In [1]:
import pandas as pd

In [2]:
train_df = pd.read_pickle("train_df_clean.pkl")
test_df = pd.read_pickle('test_df_clean.pkl')
test_df_id_col = pd.read_pickle("test_df_id_col.pkl")

In [3]:
x_train = train_df.drop(columns="trip_duration")
y_train = train_df["trip_duration"]

In [4]:
x_test = test_df

In [5]:
x_train

,vendor_id,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_distance,pickup_day_of_week,pickup_hour,is_rush_hour
0,2,1,-73.982155,40.767937,-73.964630,40.765602,0.019859,0,17,False
1,1,1,-73.980415,40.738564,-73.999481,40.731152,0.026478,6,0,False
2,2,1,-73.979027,40.763939,-74.005333,40.710087,0.080158,1,11,False
3,2,1,-74.010040,40.719971,-74.012268,40.706718,0.015480,2,19,False
4,2,1,-73.973053,40.793209,-73.972923,40.782520,0.010818,5,13,False
...,...,...,...,...,...,...,...,...,...,...
1458639,2,4,-73.982201,40.745522,-73.994911,40.740170,0.018063,4,13,False
1458640,1,1,-74.000946,40.747379,-73.970184,40.796547,0.079929,6,7,False
1458641,2,1,-73.959129,40.768799,-74.004433,40.707371,0.106731,4,6,False
1458642,1,1,-73.982079,40.749062,-73.974632,40.757107,0.015491,1,15,True


In [6]:
x_test

,vendor_id,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_distance,pickup_day_of_week,pickup_hour,is_rush_hour
0,1,1,-73.988129,40.732029,-73.990173,40.756680,0.026695,3,23,False
1,1,1,-73.964203,40.679993,-73.959808,40.655403,0.028984,3,23,False
2,1,1,-73.997437,40.737583,-73.986160,40.729523,0.019337,3,23,False
3,2,1,-73.956070,40.771900,-73.986427,40.730469,0.071789,3,23,False
4,1,1,-73.970215,40.761475,-73.961510,40.755890,0.014290,3,23,False
...,...,...,...,...,...,...,...,...,...,...
625129,1,1,-74.003464,40.725105,-74.001251,40.733643,0.010750,4,0,False
625130,1,1,-74.006363,40.743782,-73.953407,40.782467,0.091640,4,0,False
625131,1,2,-73.972267,40.759865,-73.876602,40.748665,0.106865,4,0,False
625132,1,1,-73.976501,40.733562,-73.854263,40.891788,0.280464,4,0,False


In [7]:
test_df_id_col

0         id3004672
1         id3505355
2         id1217141
3         id2150126
4         id1598245
            ...    
625129    id3008929
625130    id3700764
625131    id2568735
625132    id1384355
625133    id0621643
Name: id, Length: 625134, dtype: object

# Modeling

### Stacking

In [8]:
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, cross_val_predict

In [9]:
lr = LinearRegression()

In [10]:
kf = KFold(n_splits=5, shuffle=True, random_state=69)

In [11]:
y_train_pred = cross_val_predict(lr, x_train, y_train, cv=kf)
y_train_pred

array([ 611.06797378,  602.20061824, 1264.9377831 , ..., 1462.98594901,
        704.58185841,  685.2426975 ], shape=(1447358,))

In [12]:
x_train_stacked = x_train.copy()

x_train_stacked["lr_pred"] = y_train_pred

In [13]:
xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective = "reg:squarederror",
    random_state=69
)

In [14]:
xgb.fit(x_train_stacked, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

### Predicting

In [15]:
lr.fit(x_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [16]:
y_test_pred = lr.predict(x_test)

In [17]:
x_test_stacked = x_test.copy()

x_test_stacked["lr_pred"] = y_test_pred

In [18]:
y_pred = xgb.predict(x_test_stacked)

In [19]:
submission_df = pd.DataFrame({"id":test_df_id_col, "trip_duration":y_pred})

submission_df

,id,trip_duration
0,id3004672,837.884949
1,id3505355,577.931702
2,id1217141,474.646698
3,id2150126,964.609192
4,id1598245,353.329956
...,...,...
625129,id3008929,294.772797
625130,id3700764,1259.450928
625131,id2568735,1648.865845
625132,id1384355,1901.261475


In [20]:
submission_df.to_csv("stacked_submission.csv", index=False)